In [3]:
# -*- coding: utf-8 -*-
"""
rq1_reproducibility.py

Reproduce the RQ1 observation numbers from MainDataset.csv.

Expected input:
    C:\Android Mobile App\ICST2026_Ext\MainDataset.csv

Outputs:
    rq1_outputs/
        rq1_obs11_verdict_control_summary.csv
        rq1_obs12_ttfts_fallback_validation.csv
        rq1_obs13_attempt_control_summary.csv
        rq1_obs14_robust_dispersion.csv
        rq1_obs14_signature_candidates.csv
        rq1_reproducibility_summary.txt

This script is aligned with the current paper:
- analytical dataset: four emulator styles only
- controller: verdict-complete + first attempt
- verdict-complete uses instru_conclusion
- TTFTS uses study_ttfts_seconds if present, else ttfts_seconds
- direct TTFTS uses study_ttfts_direct_seconds if present
- fallback TTFTS uses study_ttfts_fallback_seconds if present, else S2_time_to_first_instru_seconds
- instrumentation window uses study_instru_test_window_seconds if present, else core_instru_window_seconds
- robustness uses signature_hash_base + Robust
"""

from __future__ import annotations

import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr, fisher_exact

# ============================================================
# CONFIG
# ============================================================

ROOT = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_FILE = ROOT / "MainDataset.csv"
OUT_DIR = ROOT / "rq1_outputs"

IN_SCOPE_STYLES = {"Emu_Community", "Emu_Custom", "GMD", "Third-Party"}
VERDICT_COMPLETE = {"success", "failure"}

# ============================================================
# HELPERS
# ============================================================

def pct(n, d):
    return 100.0 * n / d if d else np.nan

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def med(s):
    s = to_num(s).dropna()
    return s.median() if len(s) else np.nan

def p95(s):
    s = to_num(s).dropna()
    return s.quantile(0.95) if len(s) else np.nan

def p95_over_med(s):
    s = to_num(s).dropna()
    if len(s) == 0:
        return np.nan
    m = s.median()
    if pd.isna(m) or m == 0:
        return np.nan
    return s.quantile(0.95) / m

def iqr_over_med(s):
    s = to_num(s).dropna()
    if len(s) == 0:
        return np.nan
    m = s.median()
    if pd.isna(m) or m == 0:
        return np.nan
    return (s.quantile(0.75) - s.quantile(0.25)) / m

def cliffs_delta(x, y):
    x = to_num(pd.Series(x)).dropna().to_numpy()
    y = to_num(pd.Series(y)).dropna().to_numpy()
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))

def mw_p_delta(x, y):
    x = to_num(pd.Series(x)).dropna()
    y = to_num(pd.Series(y)).dropna()
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan
    stat = mannwhitneyu(x, y, alternative="two-sided")
    return stat.pvalue, cliffs_delta(x, y)

def fmt_p(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-50:
        return "<1e-50"
    if p < 1e-40:
        return "<1e-40"
    if p < 1e-30:
        return "<1e-30"
    if p < 1e-20:
        return "<1e-20"
    if p < 1e-10:
        return "<1e-10"
    return f"{p:.3g}"

def choose_col(df, preferred, fallback=None, required=True):
    for c in preferred:
        if c in df.columns:
            return c
    if fallback:
        for c in fallback:
            if c in df.columns:
                return c
    if required:
        raise ValueError(f"None of these columns found: {preferred + (fallback or [])}")
    return None

def print_section(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(IN_FILE, low_memory=False)

# Resolve study-facing columns
STYLE_COL = choose_col(df, ["style"])
ATTEMPT_COL = choose_col(df, ["run_attempt"])
CONCL_COL = choose_col(df, ["instru_conclusion"])
RUN_COL = choose_col(df, ["study_run_duration_seconds", "run_duration_seconds"])
TTFTS_COL = choose_col(df, ["study_ttfts_seconds", "ttfts_seconds"])
TTFTS_DIRECT_COL = choose_col(
    df,
    ["study_ttfts_direct_seconds"],
    ["time_to_first_instru_from_anchor_job_seconds", "ttfts_seconds"],
    required=False,
)
TTFTS_FALLBACK_COL = choose_col(
    df,
    ["study_ttfts_fallback_seconds"],
    ["S2_time_to_first_instru_seconds"],
    required=False,
)
WINDOW_COL = choose_col(
    df,
    ["study_instru_test_window_seconds", "core_instru_window_seconds"],
    required=False,
)
ROBUST_COL = choose_col(df, ["Robust"])
SIG_COL = choose_col(df, ["signature_hash_base"])

style_map = {
    "Emu_Community": "Community",
    "Emu_Custom": "Custom",
    "GMD": "GMD",
    "Third-Party": "Third-Party",
}

# Four-style emulator analytical dataset
emu = df[df[STYLE_COL].isin(IN_SCOPE_STYLES)].copy()
emu["style_pretty"] = emu[STYLE_COL].map(style_map)
emu["verdict_complete"] = emu[CONCL_COL].fillna("").astype(str).str.lower().isin(VERDICT_COMPLETE)
emu["first_attempt"] = to_num(emu[ATTEMPT_COL]).fillna(1).eq(1)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# OBS 1.1 -- VERDICT-COMPLETE VS NON-VERDICT
# ============================================================

verdict = emu[emu["verdict_complete"]].copy()
non_verdict = emu[~emu["verdict_complete"]].copy()

obs11_rows = []
for regime_name, sub in [("verdict-complete", verdict), ("non-verdict", non_verdict)]:
    obs11_rows.append({
        "Regime": regime_name,
        "n": len(sub),
        "run_med_s": med(sub[RUN_COL]),
        "run_p95_over_med": p95_over_med(sub[RUN_COL]),
        "ttfts_med_s": med(sub[TTFTS_COL]),
        "ttfts_p95_over_med": p95_over_med(sub[TTFTS_COL]),
        "window_med_s": med(sub[WINDOW_COL]) if WINDOW_COL else np.nan,
        "window_p95_over_med": p95_over_med(sub[WINDOW_COL]) if WINDOW_COL else np.nan,
    })

obs11_df = pd.DataFrame(obs11_rows)
obs11_df.to_csv(OUT_DIR / "rq1_obs11_verdict_control_summary.csv", index=False)

run_p_11, run_d_11 = mw_p_delta(verdict[RUN_COL], non_verdict[RUN_COL])
ttfts_p_11, ttfts_d_11 = mw_p_delta(verdict[TTFTS_COL], non_verdict[TTFTS_COL])
if WINDOW_COL:
    win_p_11, win_d_11 = mw_p_delta(verdict[WINDOW_COL], non_verdict[WINDOW_COL])
else:
    win_p_11, win_d_11 = np.nan, np.nan

# ============================================================
# OBS 1.2 -- TTFTS FALLBACK VALIDATION
# ============================================================

ttfts_available = emu[TTFTS_COL].notna()

if TTFTS_DIRECT_COL is None:
    direct_mask = pd.Series(False, index=emu.index)
else:
    direct_mask = emu[TTFTS_DIRECT_COL].notna()

if TTFTS_FALLBACK_COL is None:
    fallback_mask = pd.Series(False, index=emu.index)
else:
    fallback_mask = emu[TTFTS_FALLBACK_COL].notna()

# Count direct as direct TTFTS available
# Count fallback-only as fallback available when final TTFTS exists but direct does not
direct_count = int((ttfts_available & direct_mask).sum())
fallback_count = int((ttfts_available & (~direct_mask) & fallback_mask).sum())
missing_count = int((~ttfts_available).sum())

# Comparable overlap pairs for fallback validation
if TTFTS_DIRECT_COL is not None and TTFTS_FALLBACK_COL is not None:
    overlap_pairs = emu[
        emu[TTFTS_DIRECT_COL].notna()
        & emu[TTFTS_FALLBACK_COL].notna()
    ].copy()
else:
    overlap_pairs = emu.iloc[0:0].copy()

diff = to_num(overlap_pairs[TTFTS_DIRECT_COL]) - to_num(overlap_pairs[TTFTS_FALLBACK_COL])
rho, rho_p = (np.nan, np.nan)
if len(overlap_pairs) > 1:
    rho, rho_p = spearmanr(
        to_num(overlap_pairs[TTFTS_DIRECT_COL]),
        to_num(overlap_pairs[TTFTS_FALLBACK_COL]),
        nan_policy="omit"
    )

obs12_df = pd.DataFrame([{
    "n_total": len(emu),
    "ttfts_available_n": int(ttfts_available.sum()),
    "ttfts_available_pct": pct(int(ttfts_available.sum()), len(emu)),
    "direct_n": direct_count,
    "direct_pct_of_available": pct(direct_count, int(ttfts_available.sum())),
    "fallback_n": fallback_count,
    "fallback_pct_of_available": pct(fallback_count, int(ttfts_available.sum())),
    "missing_n": missing_count,
    "overlap_pairs_n": len(overlap_pairs),
    "direct_med_s": med(overlap_pairs[TTFTS_DIRECT_COL]) if len(overlap_pairs) else np.nan,
    "fallback_med_s": med(overlap_pairs[TTFTS_FALLBACK_COL]) if len(overlap_pairs) else np.nan,
    "median_diff_s": med(diff) if len(overlap_pairs) else np.nan,
    "median_abs_err_s": med(diff.abs()) if len(overlap_pairs) else np.nan,
    "mean_diff_s": diff.mean() if len(overlap_pairs) else np.nan,
    "spearman_rho": rho,
    "spearman_p": rho_p,
    "exact_match_pct": pct(int((diff == 0).sum()), len(diff)) if len(diff) else np.nan,
    "within_30s_pct": pct(int((diff.abs() <= 30).sum()), len(diff)) if len(diff) else np.nan,
}])
obs12_df.to_csv(OUT_DIR / "rq1_obs12_ttfts_fallback_validation.csv", index=False)

# ============================================================
# OBS 1.3 -- FIRST ATTEMPT VS RERUN
# ============================================================

attempt1 = emu[emu["first_attempt"]].copy()
rerun = emu[~emu["first_attempt"]].copy()

a = int(attempt1["verdict_complete"].sum())
b = int((~attempt1["verdict_complete"]).sum())
c = int(rerun["verdict_complete"].sum())
d = int((~rerun["verdict_complete"]).sum())
_, fisher_p = fisher_exact([[a, b], [c, d]], alternative="two-sided")

attempt1_v = attempt1[attempt1["verdict_complete"]].copy()
rerun_v = rerun[rerun["verdict_complete"]].copy()

obs13_df = pd.DataFrame([
    {
        "Attempt": "attempt=1",
        "n": len(attempt1),
        "verdict_complete_n": a,
        "verdict_complete_pct": pct(a, len(attempt1)),
        "run_med_s": med(attempt1_v[RUN_COL]),
        "run_p95_over_med": p95_over_med(attempt1_v[RUN_COL]),
        "ttfts_med_s": med(attempt1_v[TTFTS_COL]),
        "ttfts_p95_over_med": p95_over_med(attempt1_v[TTFTS_COL]),
        "window_med_s": med(attempt1_v[WINDOW_COL]) if WINDOW_COL else np.nan,
        "window_p95_over_med": p95_over_med(attempt1_v[WINDOW_COL]) if WINDOW_COL else np.nan,
    },
    {
        "Attempt": "attempt>1",
        "n": len(rerun),
        "verdict_complete_n": c,
        "verdict_complete_pct": pct(c, len(rerun)),
        "run_med_s": med(rerun_v[RUN_COL]),
        "run_p95_over_med": p95_over_med(rerun_v[RUN_COL]),
        "ttfts_med_s": med(rerun_v[TTFTS_COL]),
        "ttfts_p95_over_med": p95_over_med(rerun_v[TTFTS_COL]),
        "window_med_s": med(rerun_v[WINDOW_COL]) if WINDOW_COL else np.nan,
        "window_p95_over_med": p95_over_med(rerun_v[WINDOW_COL]) if WINDOW_COL else np.nan,
    },
])
obs13_df.to_csv(OUT_DIR / "rq1_obs13_attempt_control_summary.csv", index=False)

run_p_13, run_d_13 = mw_p_delta(attempt1_v[RUN_COL], rerun_v[RUN_COL])
ttfts_p_13, ttfts_d_13 = mw_p_delta(attempt1_v[TTFTS_COL], rerun_v[TTFTS_COL])
if WINDOW_COL:
    win_p_13, win_d_13 = mw_p_delta(attempt1_v[WINDOW_COL], rerun_v[WINDOW_COL])
else:
    win_p_13, win_d_13 = np.nan, np.nan

# ============================================================
# OBS 1.4 -- ROBUSTNESS LAYER
# ============================================================

ctrl = emu[emu["verdict_complete"] & emu["first_attempt"]].copy()
robust = ctrl[ctrl[ROBUST_COL] == True].copy()

disp_rows = []
for label, col in [
    ("Run duration", RUN_COL),
    ("TTFTS", TTFTS_COL),
    ("Instrumentation window", WINDOW_COL),
]:
    if col is None:
        continue
    base_disp = iqr_over_med(ctrl[col])
    robust_disp = iqr_over_med(robust[col])
    red_pct = ((base_disp - robust_disp) / base_disp * 100.0) if pd.notna(base_disp) and base_disp != 0 else np.nan
    disp_rows.append({
        "Measure": label,
        "Base_IQR_over_Median": base_disp,
        "Robust_IQR_over_Median": robust_disp,
        "Reduction_pct": red_pct,
    })

obs14_disp_df = pd.DataFrame(disp_rows)
obs14_disp_df.to_csv(OUT_DIR / "rq1_obs14_robust_dispersion.csv", index=False)

sig_counts = (
    robust.groupby([SIG_COL, "style_pretty"])
    .size()
    .unstack(fill_value=0)
)

for col in ["Community", "Custom", "GMD", "Third-Party"]:
    if col not in sig_counts.columns:
        sig_counts[col] = 0

sig_counts["Total_n"] = sig_counts[["Community", "Custom", "GMD", "Third-Party"]].sum(axis=1)
sig_counts["n_styles_present"] = (sig_counts[["Community", "Custom", "GMD", "Third-Party"]] > 0).sum(axis=1)
sig_counts["has_comm_gmd_tp"] = (
    (sig_counts["Community"] > 0)
    & (sig_counts["GMD"] > 0)
    & (sig_counts["Third-Party"] > 0)
)

sig_rank = sig_counts.reset_index().sort_values(
    by=["has_comm_gmd_tp", "n_styles_present", "Total_n", "Community", "GMD", "Third-Party"],
    ascending=[False, False, False, False, False, False]
)

sig_rank.to_csv(OUT_DIR / "rq1_obs14_signature_candidates.csv", index=False)

# ============================================================
# SUMMARY TXT
# ============================================================

summary_lines = []
summary_lines.append("RQ1 REPRODUCIBILITY SUMMARY")
summary_lines.append("=" * 60)
summary_lines.append(f"Analytical four-style emulator dataset n = {len(emu):,}")
summary_lines.append("")
summary_lines.append("Obs. 1.1 -- verdict-complete vs non-verdict")
summary_lines.append(f"  verdict-complete n = {len(verdict):,} ({pct(len(verdict), len(emu)):.1f}%)")
summary_lines.append(f"  non-verdict n      = {len(non_verdict):,} ({pct(len(non_verdict), len(emu)):.1f}%)")
summary_lines.append(f"  Run duration MW p = {fmt_p(run_p_11)}, Cliff's delta = {run_d_11:.3f}")
summary_lines.append(f"  TTFTS MW p        = {fmt_p(ttfts_p_11)}, Cliff's delta = {ttfts_d_11:.3f}")
if WINDOW_COL:
    summary_lines.append(f"  Window MW p       = {fmt_p(win_p_11)}, Cliff's delta = {win_d_11:.3f}")
summary_lines.append("")
summary_lines.append("Obs. 1.2 -- TTFTS fallback validation")
summary_lines.append(f"  TTFTS available   = {int(ttfts_available.sum()):,} / {len(emu):,} ({pct(int(ttfts_available.sum()), len(emu)):.1f}%)")
summary_lines.append(f"  direct n          = {direct_count:,}")
summary_lines.append(f"  fallback n        = {fallback_count:,}")
summary_lines.append(f"  missing n         = {missing_count:,}")
summary_lines.append(f"  overlap pairs n   = {len(overlap_pairs):,}")
summary_lines.append(f"  Spearman rho      = {rho:.3f}" if pd.notna(rho) else "  Spearman rho      = NA")
summary_lines.append("")
summary_lines.append("Obs. 1.3 -- first attempt vs rerun")
summary_lines.append(f"  attempt=1 n       = {len(attempt1):,}")
summary_lines.append(f"  attempt>1 n       = {len(rerun):,}")
summary_lines.append(f"  Fisher exact p    = {fmt_p(fisher_p)}")
summary_lines.append(f"  Run duration MW p = {fmt_p(run_p_13)}, Cliff's delta = {run_d_13:.3f}")
summary_lines.append(f"  TTFTS MW p        = {fmt_p(ttfts_p_13)}, Cliff's delta = {ttfts_d_13:.3f}")
if WINDOW_COL:
    summary_lines.append(f"  Window MW p       = {fmt_p(win_p_13)}, Cliff's delta = {win_d_13:.3f}")
summary_lines.append("")
summary_lines.append("Obs. 1.4 -- robustness layer")
summary_lines.append(f"  controller-filtered n = {len(ctrl):,}")
summary_lines.append(f"  robust n              = {len(robust):,}")
summary_lines.append(f"  base signatures       = {ctrl[SIG_COL].nunique():,}")
summary_lines.append(f"  robust signatures     = {robust[SIG_COL].nunique():,}")

with open(OUT_DIR / "rq1_reproducibility_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# ============================================================
# CONSOLE PRINT
# ============================================================

print_section("RQ1 REPRODUCIBILITY PACKAGE")
print("\n".join(summary_lines))
print_section("FILES WRITTEN")
for name in [
    "rq1_obs11_verdict_control_summary.csv",
    "rq1_obs12_ttfts_fallback_validation.csv",
    "rq1_obs13_attempt_control_summary.csv",
    "rq1_obs14_robust_dispersion.csv",
    "rq1_obs14_signature_candidates.csv",
    "rq1_reproducibility_summary.txt",
]:
    print(str(OUT_DIR / name))


RQ1 REPRODUCIBILITY PACKAGE
RQ1 REPRODUCIBILITY SUMMARY
Analytical four-style emulator dataset n = 9,063

Obs. 1.1 -- verdict-complete vs non-verdict
  verdict-complete n = 7,952 (87.7%)
  non-verdict n      = 1,111 (12.3%)
  Run duration MW p = <1e-30, Cliff's delta = 0.216
  TTFTS MW p        = <1e-10, Cliff's delta = -0.163
  Window MW p       = <1e-30, Cliff's delta = 0.305

Obs. 1.2 -- TTFTS fallback validation
  TTFTS available   = 8,927 / 9,063 (98.5%)
  direct n          = 8,927
  fallback n        = 0
  missing n         = 136
  overlap pairs n   = 8,925
  Spearman rho      = 0.763

Obs. 1.3 -- first attempt vs rerun
  attempt=1 n       = 8,783
  attempt>1 n       = 280
  Fisher exact p    = 0.138
  Run duration MW p = <1e-50, Cliff's delta = -0.559
  TTFTS MW p        = 0.849, Cliff's delta = -0.010
  Window MW p       = 7.79e-07, Cliff's delta = -0.276

Obs. 1.4 -- robustness layer
  controller-filtered n = 7,698
  robust n              = 6,266
  base signatures       = 29
